In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 1


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.5978553742170334
Epoch 2/100, Loss: 2.5701739862561226
Epoch 3/100, Loss: 2.5223330929875374
Epoch 4/100, Loss: 2.6874880641698837
Epoch 5/100, Loss: 2.5918658524751663
Epoch 6/100, Loss: 2.764392450451851
Epoch 7/100, Loss: 2.554467089474201
Epoch 8/100, Loss: 2.56683087348938
Epoch 9/100, Loss: 2.661341220140457
Epoch 10/100, Loss: 2.6067812219262123
Epoch 11/100, Loss: 2.5758905336260796
Epoch 12/100, Loss: 2.5233954787254333
Epoch 13/100, Loss: 2.6408369094133377
Epoch 14/100, Loss: 2.5808050110936165
Epoch 15/100, Loss: 2.650073178112507
Epoch 16/100, Loss: 2.593384772539139
Epoch 17/100, Loss: 2.532645121216774


Epoch 18/100, Loss: 2.6037988290190697
Epoch 19/100, Loss: 2.516862116754055
Epoch 20/100, Loss: 2.671688497066498
Epoch 21/100, Loss: 2.597827598452568
Epoch 22/100, Loss: 2.6977747455239296
Epoch 23/100, Loss: 2.4480183348059654
Epoch 24/100, Loss: 2.557854577898979
Epoch 25/100, Loss: 2.4833518341183662
Epoch 26/100, Loss: 2.572628654539585
Epoch 27/100, Loss: 2.5557193905115128
Epoch 28/100, Loss: 2.7147995308041573
Epoch 29/100, Loss: 2.58649979531765
Epoch 30/100, Loss: 2.6399931982159615
Epoch 31/100, Loss: 2.7250454276800156


Epoch 32/100, Loss: 2.603483185172081
Epoch 33/100, Loss: 2.585220128297806
Epoch 34/100, Loss: 2.67693629860878
Epoch 35/100, Loss: 2.603554427623749
Epoch 36/100, Loss: 2.661882109940052
Epoch 37/100, Loss: 2.6167358309030533
Epoch 38/100, Loss: 2.502002216875553
Epoch 39/100, Loss: 2.787282131612301
Epoch 40/100, Loss: 2.65914586186409
Epoch 41/100, Loss: 2.528372645378113
Epoch 42/100, Loss: 2.582917094230652
Epoch 43/100, Loss: 2.5529686510562897


Epoch 44/100, Loss: 2.545245297253132
Epoch 45/100, Loss: 2.6335377246141434
Epoch 46/100, Loss: 2.488495096564293
Epoch 47/100, Loss: 2.5992800295352936
Epoch 48/100, Loss: 2.5166738107800484
Epoch 49/100, Loss: 2.6619278863072395
Epoch 50/100, Loss: 2.6787101477384567
Epoch 51/100, Loss: 2.393850952386856
Epoch 52/100, Loss: 2.5380784571170807
Epoch 53/100, Loss: 2.6131972447037697
Epoch 54/100, Loss: 2.5839807763695717
Epoch 55/100, Loss: 2.587231360375881
Epoch 56/100, Loss: 2.580062784254551
Epoch 57/100, Loss: 2.648818925023079


Epoch 58/100, Loss: 2.576160356402397
Epoch 59/100, Loss: 2.575393468141556
Epoch 60/100, Loss: 2.7617015466094017
Epoch 61/100, Loss: 2.6056589633226395
Epoch 62/100, Loss: 2.540405161678791
Epoch 63/100, Loss: 2.7455796524882317
Epoch 64/100, Loss: 2.6250236704945564
Epoch 65/100, Loss: 2.566180557012558
Epoch 66/100, Loss: 2.6509545370936394
Epoch 67/100, Loss: 2.6148820519447327
Epoch 68/100, Loss: 2.5552605390548706
Epoch 69/100, Loss: 2.5970800295472145
Epoch 70/100, Loss: 2.667554423213005
Epoch 71/100, Loss: 2.5260137170553207
Epoch 72/100, Loss: 2.589924566447735
Epoch 73/100, Loss: 2.4482452273368835


...

Epoch 38/100, Loss: 1.807386189699173
Epoch 39/100, Loss: 1.7638419643044472
Epoch 40/100, Loss: 1.7357196062803268
Epoch 41/100, Loss: 1.7889071255922318
Epoch 42/100, Loss: 1.7892185747623444
Epoch 43/100, Loss: 1.7955934554338455
Epoch 44/100, Loss: 1.859212450683117
Epoch 45/100, Loss: 1.7827570736408234
Epoch 46/100, Loss: 1.7524181082844734
Epoch 47/100, Loss: 1.7726173251867294
Epoch 48/100, Loss: 1.8707883059978485
Epoch 49/100, Loss: 1.83320152759552
Epoch 50/100, Loss: 1.746676567941904
Epoch 51/100, Loss: 1.7732801586389542
Epoch 52/100, Loss: 1.7340330258011818
Epoch 53/100, Loss: 1.5665863752365112
Epoch 54/100, Loss: 1.7478958405554295


Epoch 55/100, Loss: 1.7186877354979515
Epoch 56/100, Loss: 1.888500228524208
Epoch 57/100, Loss: 1.7843251153826714
Epoch 58/100, Loss: 2.0168623626232147
Epoch 59/100, Loss: 1.968844436109066
Epoch 60/100, Loss: 1.6896231696009636
Epoch 61/100, Loss: 1.7176074832677841
Epoch 62/100, Loss: 1.7713027074933052
Epoch 63/100, Loss: 1.8133211210370064
Epoch 64/100, Loss: 1.866812601685524
Epoch 65/100, Loss: 1.7656779289245605
Epoch 66/100, Loss: 1.7854989543557167
Epoch 67/100, Loss: 1.846178650856018
Epoch 68/100, Loss: 1.8227030858397484


Epoch 69/100, Loss: 1.8822631686925888
Epoch 70/100, Loss: 2.0660286471247673
Epoch 71/100, Loss: 1.874046877026558
Epoch 72/100, Loss: 1.8572660088539124
Epoch 73/100, Loss: 1.8613182082772255
Epoch 74/100, Loss: 1.7234709784388542
Epoch 75/100, Loss: 1.8159572929143906
Epoch 76/100, Loss: 1.8357220143079758
Epoch 77/100, Loss: 1.6524041816592216
Epoch 78/100, Loss: 1.6691348850727081
Epoch 79/100, Loss: 1.784796066582203
Epoch 80/100, Loss: 1.8777738884091377
Epoch 81/100, Loss: 1.8177912160754204
Epoch 82/100, Loss: 1.6400406211614609
Epoch 83/100, Loss: 1.6829278022050858
Epoch 84/100, Loss: 1.8645168393850327


Epoch 85/100, Loss: 1.8190896734595299
Epoch 86/100, Loss: 1.7971790879964828
Epoch 87/100, Loss: 1.8402887061238289
Epoch 88/100, Loss: 1.8027028366923332
Epoch 89/100, Loss: 1.8608281761407852
Epoch 90/100, Loss: 1.717595174908638
Epoch 91/100, Loss: 1.7499251961708069
Epoch 92/100, Loss: 1.670396476984024
Epoch 93/100, Loss: 1.8306901082396507
Epoch 94/100, Loss: 1.790362760424614
Epoch 95/100, Loss: 1.7438544854521751
Epoch 96/100, Loss: 1.8269555419683456
Epoch 97/100, Loss: 1.7907740026712418
Epoch 98/100, Loss: 1.843907169997692


Epoch 99/100, Loss: 1.8050856813788414
Epoch 100/100, Loss: 1.7676104232668877
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.3626
